# 01 · Consolidação dos alunos, grupos e verbetes

As planilhas acadêmicas têm formatos diferentes. Aqui organizamos os campos comuns e aplicamos as correções documentadas na base auditada. Não fazemos associação automática por semelhança de nomes sem conferência.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
BRUTOS = ROOT / 'dados' / 'brutos'
PROCESSADOS = ROOT / 'dados' / 'processados'
GRAFICOS = ROOT / 'resultados' / 'graficos'
PROCESSADOS.mkdir(parents=True, exist_ok=True)
GRAFICOS.mkdir(parents=True, exist_ok=True)

## Fontes originais

In [ ]:
alunos = pd.read_excel(BRUTOS / 'Wikipedia Alunos (1).xlsx', sheet_name='Wikipedia')
verbetes = pd.read_excel(BRUTOS / 'Tabela de Verbetes (1).xlsx', sheet_name='Sheet2')
cruzamento = pd.read_excel(BRUTOS / 'Cruzamento_Wikipedia.xlsx')
print('Alunos da planilha original:', len(alunos))
print('Linhas de grupos/verbetes:', len(verbetes))
print('Cruzamentos documentados:', len(cruzamento))
display(alunos.head(3))
display(verbetes.iloc[:3, :11])

## Padronização inicial

O cadastro de grupos tem integrantes distribuídos por várias colunas. Colocamos um integrante por linha para facilitar os cruzamentos. O resultado automático serve à conferência; títulos e usuários corrigidos são mantidos na base auditada.

In [ ]:
from unicodedata import normalize
import re

def chave_texto(valor):
    if pd.isna(valor):
        return ''
    txt = normalize('NFKD', str(valor)).encode('ascii', 'ignore').decode('ascii')
    return re.sub(r'\s+', ' ', txt).strip().casefold()

campos = list(verbetes.columns)
campos_nomes = campos[7:]  # colunas após Ano, Período, Disciplina, Verbete, Link, Observação e Grupo
membros = verbetes.melt(
    id_vars=campos[:7], value_vars=campos_nomes,
    var_name='coluna_integrante', value_name='nome_aluno'
)
membros = membros.dropna(subset=['nome_aluno']).copy()
membros['nome_aluno'] = membros['nome_aluno'].astype(str).str.strip()
membros = membros[membros['nome_aluno'].ne('')]
membros['chave_nome'] = membros['nome_aluno'].map(chave_texto)
alunos['chave_nome'] = alunos['Nome do Aluno'].map(chave_texto)
print('Registros de integrantes listados:', len(membros))
display(membros.head(5))

## Ajustes auditados

A planilha `df_tcc_base_consolidada_corrigida.xlsx` guarda as decisões de cruzamento feitas na pesquisa: nomes de usuário, verbete na API, disciplina, período e casos ambíguos. Ela é a referência para a base final, e não um cruzamento aproximado por nome.

In [ ]:
auditado = BRUTOS / 'df_tcc_base_consolidada_corrigida.xlsx'
base = pd.read_excel(auditado, sheet_name='Base_Consolidada')
mapa = pd.read_excel(auditado, sheet_name='Mapa_Grupos')
auditoria = pd.read_excel(auditado, sheet_name='Auditoria')

base['reprocessar_api'] = base['reprocessar_api'].astype(str).str.strip().str.upper()
aptas = base.loc[base['reprocessar_api'].eq('SIM')].copy().reset_index(drop=True)
aptas.insert(0, 'id_participacao', [f'P{i:04d}' for i in range(1, len(aptas)+1)])

colunas_base = [
    'id_participacao', 'ano', 'grupo', 'turma', 'tipo_participacao',
    'nome_aluno', 'usuario_wikipedia', 'verbete_projeto', 'verbete', 'wiki_api',
    'observacao_base', 'reprocessar_api', 'status_preparo'
]
for col in colunas_base:
    if col not in aptas:
        aptas[col] = pd.NA
aptas = aptas[colunas_base]

assert len(aptas) == 243, f'Eram esperadas 243 participações aptas; encontramos {len(aptas)}.'
assert aptas['id_participacao'].is_unique
print('Participações aptas à API:',len(aptas))
print('Por wiki:',aptas['wiki_api'].value_counts(dropna=False).to_dict())

## Gravar planilha consolidada

As outras etapas usam a aba `Base_Consolidada`. As abas de mapa e auditoria documentam as decisões manuais.

In [ ]:
saida = PROCESSADOS / 'base_participacoes_consolidada.xlsx'
with pd.ExcelWriter(saida, engine='openpyxl') as writer:
    aptas.to_excel(writer, sheet_name='Base_Consolidada', index=False)
    mapa.to_excel(writer, sheet_name='Mapa_Grupos', index=False)
    auditoria.to_excel(writer, sheet_name='Auditoria', index=False)
print('Arquivo gerado:',saida.name)